In [16]:
# =========================
# JSON Refiner Advanced Edition (Full Session)
# =========================

!pip install -q gradio jsonschema pyyaml pandas==2.2.2 transformers accelerate

import os
import re
import json
import yaml
import tempfile
from copy import deepcopy
from datetime import datetime
import pandas as pd
import gradio as gr
from jsonschema import Draft7Validator

MAX_JSON_SIZE = 5_000_000

In [2]:
# --------------------------
# Optional Granite model
# --------------------------
ai_pipe = None

def load_granite():
    global ai_pipe
    if ai_pipe is not None:
        return "Granite already loaded."
    try:
        from transformers import pipeline
        ai_pipe = pipeline(
            "text-generation",
            model="ibm-granite/granite-3.3-2b-instruct",
            device_map="auto"
        )
        return "Granite model loaded."
    except Exception as e:
        return f"Granite load failed: {e}"


def granite_summary(data, enabled):
    if not enabled:
        return "AI summary disabled."

    if ai_pipe is None:
        return "Load Granite model first."

    try:
        prompt = f"""
You are a JSON expert.

Analyze the JSON and provide:

1. Short summary
2. Possible issues
3. Three improvements

JSON:
{json.dumps(data)[:4000]}
"""

        out = ai_pipe(prompt, max_new_tokens=200)
        return out[0]["generated_text"]

    except Exception as e:
        return f"AI summary error: {e}"


In [3]:
# --------------------------
# Text helpers
# --------------------------
def split_words(s):
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", str(s))
    s = s.replace("_", " ").replace("-", " ")
    return [w.lower() for w in s.split() if w.strip()]


def to_case(key, mode):
    w = split_words(key)

    if mode == "keep" or not w:
        return key

    if mode == "snake_case":
        return "_".join(w)

    if mode == "camelCase":
        return w[0] + "".join(x.capitalize() for x in w[1:])

    if mode == "PascalCase":
        return "".join(x.capitalize() for x in w)

    if mode == "kebab-case":
        return "-".join(w)

    return key


def convert_keys(obj, mode):
    if isinstance(obj, dict):
        return {to_case(k, mode): convert_keys(v, mode) for k, v in obj.items()}

    if isinstance(obj, list):
        return [convert_keys(x, mode) for x in obj]

    return obj


In [4]:
# --------------------------
# KV Parser
# --------------------------
def parse_kv_text(text):
    out = {}
    errs = []

    for i, line in enumerate(text.splitlines(), 1):
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        if ":" not in line:
            errs.append(f"Line {i}: missing ':'")
            continue

        k, v = line.split(":", 1)

        k = k.strip()
        v = v.strip()

        if not k:
            errs.append(f"Line {i}: empty key")
            continue

        out[k] = infer_type(v)

    return out, errs


def infer_type(v):

    t = v.strip().lower()

    if t in ["null", "none", "nil", "undefined", "na"]:
        return None

    if t in ["true", "yes", "on"]:
        return True

    if t in ["false", "no", "off"]:
        return False

    if re.fullmatch(r"-?\d+", v):
        return int(v)

    if re.fullmatch(r"-?\d+\.\d+", v):
        return float(v)

    try:
        if v.startswith("{") or v.startswith("["):
            return json.loads(v)
    except:
        pass

    return v




In [5]:
# --------------------------
# Flatten / Unflatten
# --------------------------
def flatten_json(data, parent="", sep="."):

    out = {}

    if isinstance(data, dict):

        for k, v in data.items():

            nk = f"{parent}{sep}{k}" if parent else k
            out.update(flatten_json(v, nk, sep))

    elif isinstance(data, list):

        for i, v in enumerate(data):

            nk = f"{parent}{sep}{i}" if parent else str(i)
            out.update(flatten_json(v, nk, sep))

    else:

        out[parent] = data

    return out


def unflatten_json(flat, sep="."):

    root = {}

    for key, value in flat.items():

        parts = key.split(sep)

        cur = root

        for i, part in enumerate(parts):

            is_last = i == len(parts) - 1

            if part.isdigit():
                part = int(part)

            if is_last:

                if isinstance(cur, list):

                    while len(cur) <= part:
                        cur.append(None)

                    cur[part] = value

                else:
                    cur[part] = value

            else:

                next_part = parts[i + 1]

                make_list = next_part.isdigit()

                if isinstance(cur, list):

                    while len(cur) <= part:
                        cur.append([] if make_list else {})

                    if cur[part] is None:
                        cur[part] = [] if make_list else {}

                    cur = cur[part]

                else:

                    if part not in cur:
                        cur[part] = [] if make_list else {}

                    cur = cur[part]

    return root




In [6]:
# --------------------------
# Merge
# --------------------------
def deep_merge(a, b):

    if isinstance(a, dict) and isinstance(b, dict):

        out = deepcopy(a)

        for k, v in b.items():

            if k in out:
                out[k] = deep_merge(out[k], v)

            else:
                out[k] = deepcopy(v)

        return out

    if isinstance(a, list) and isinstance(b, list):

        return a + b

    return deepcopy(b)


In [7]:
# --------------------------
# Null remover
# --------------------------
def remove_nulls(x):

    if isinstance(x, dict):

        return {k: remove_nulls(v) for k, v in x.items() if v is not None}

    if isinstance(x, list):

        return [remove_nulls(i) for i in x if i is not None]

    return x


In [15]:
# --------------------------
# Schema validation
# --------------------------
def schema_validate(data, schema_text):

    if not schema_text.strip():
        return "Schema skipped."

    try:

        schema = json.loads(schema_text)

        errs = sorted(
            Draft7Validator(schema).iter_errors(data),
            key=lambda e: list(e.path),
        )

        if not errs:
            return "Schema valid."

        lines = []

        for e in errs[:10]:

            p = "/".join(map(str, e.path)) if e.path else "root"

            lines.append(f"- {p}: {e.message}")

        return "Schema invalid:\n" + "\n".join(lines)

    except Exception as e:
        return f"Schema error: {e}"




In [9]:
# --------------------------
# Schema inference
# --------------------------
def infer_schema(data):

    def t(v):

        if isinstance(v, dict):

            return {
                "type": "object",
                "properties": {k: t(x) for k, x in v.items()},
            }

        if isinstance(v, list):

            return {
                "type": "array",
                "items": t(v[0]) if v else {},
            }

        if isinstance(v, bool):
            return {"type": "boolean"}

        if isinstance(v, int):
            return {"type": "integer"}

        if isinstance(v, float):
            return {"type": "number"}

        if v is None:
            return {"type": "null"}

        return {"type": "string"}

    return {"$schema": "http://json-schema.org/draft-07/schema#", **t(data)}



In [10]:
# --------------------------
# File helper
# --------------------------
def temp_file(text, suffix, prefix):

    fd, path = tempfile.mkstemp(prefix=prefix + "_", suffix=suffix)

    with os.fdopen(fd, "w", encoding="utf-8") as f:
        f.write(text)

    return path


def read_json_file(file_obj):

    if file_obj is None:
        return gr.update(), "No file selected."
    try:
        path = file_obj if isinstance(file_obj, str) else file_obj.name
        with open(path, "r", encoding="utf-8") as f:
            return json.dumps(json.load(f), indent=2, ensure_ascii=False), "File loaded."
    except Exception as e:
        return gr.update(), f"File error: {e}"


In [11]:

# --------------------------
# History
# --------------------------
def push_history(state, action, data):

    new_state = deepcopy(state)

    new_state["items"] = new_state["items"][: new_state["idx"] + 1]

    new_state["items"].append(deepcopy(data))

    new_state["events"].append(
        {
            "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "action": action,
        }
    )

    new_state["idx"] = len(new_state["items"]) - 1

    return new_state




In [12]:
# --------------------------
# Main processing
# --------------------------
def process_all(
    input_mode,
    input_text,
    merge_text,
    schema_text,
    case_style,
    use_merge,
    do_flatten,
    do_unflatten,
    drop_nulls,
    sort_keys,
    use_ai,
    state,
):

    if len(input_text) > MAX_JSON_SIZE:
        return "", "", "", "", "", "Input JSON too large.", state, None, None, None, "[]"

    try:

        if input_mode == "JSON":

            data = json.loads(input_text)

            warnings = []

        else:

            data, warnings = parse_kv_text(input_text)

    except Exception as e:

        return "", "", "", "", "", f"Input error: {e}", state, None, None, None, "[]"

    if use_merge and merge_text.strip():

        try:

            merge_data = json.loads(merge_text)

            data = deep_merge(data, merge_data)

        except Exception as e:

            return "", "", "", "", "", f"Merge error: {e}", state, None, None, None, "[]"

    data = convert_keys(data, case_style)

    if drop_nulls:
        data = remove_nulls(data)

    if do_flatten and do_unflatten:

        return "", "", "", "", "", "Choose flatten or unflatten only.", state, None, None, None, "[]"

    if do_flatten:
        data = flatten_json(data)

    if do_unflatten:

        if not isinstance(data, dict):
            return "", "", "", "", "", "Unflatten needs dict.", state, None, None, None, "[]"

        data = unflatten_json(data)

    schema_msg = schema_validate(data, schema_text)

    ai_msg = granite_summary(data, use_ai)

    warn_msg = "\n".join(warnings[:10]) if warnings else "No parse warnings."

    out_json = json.dumps(data, indent=2, ensure_ascii=False, sort_keys=sort_keys)

    out_yaml = yaml.safe_dump(data, allow_unicode=True, sort_keys=sort_keys)

    flat = flatten_json(data)

    out_csv = pd.DataFrame(list(flat.items()), columns=["key", "value"]).to_csv(index=False)

    report = f"""
# JSON Refiner Report

Time: {datetime.now()}

Schema: {schema_msg}

Warnings:
{warn_msg}
"""

    schema_hint = json.dumps(infer_schema(data), indent=2)

    state = push_history(state, "process", data)

    history_view = json.dumps(
        {"idx": state["idx"], "events": state["events"]},
        indent=2,
    )

    out_file = temp_file(out_json, ".json", "output")

    hist_file = temp_file(json.dumps(state, indent=2), ".json", "history")

    rep_file = temp_file(report, ".md", "report")

    status = f"{schema_msg}\n\n{ai_msg}"

    return (
        out_json,
        out_yaml,
        out_csv,
        report,
        schema_hint,
        status,
        state,
        out_file,
        hist_file,
        rep_file,
        history_view,
    )



In [13]:
# --------------------------
# UI
# --------------------------
sample_json = {
    "user_name": "Rahul",
    "profile_info": {"city": "Delhi", "pin": 110001},
    "skills": ["python", "json"],
    "active": True,
    "misc": None,
}

custom_css = """
body {
    background: radial-gradient(circle at top left, #2a1f07 0%, #111111 45%, #080808 100%) !important;
    color: #f3e7c9 !important;
}

.gradio-container {
    background: linear-gradient(135deg, #1a1408 0%, #0f0f0f 100%) !important;
    border: 1px solid #6f5a1d !important;
    box-shadow: 0 0 24px rgba(212, 175, 55, 0.18) !important;
}

.input_textbox textarea,
textarea,
input,
select {
    background-color: #1f1a10 !important;
    color: #f8edcf !important;
    border: 1px solid #b08d2f !important;
    box-shadow: inset 0 0 8px rgba(212, 175, 55, 0.15) !important;
}

.output_codeblock pre,
code,
pre {
    background: #141108 !important;
    color: #f7e7bd !important;
    border: 1px solid #c9a646 !important;
    box-shadow: 0 0 12px rgba(212, 175, 55, 0.12) !important;
}

h1, h2, h3, h4 {
    color: #ffd66b !important;
    text-shadow: 0 0 8px rgba(255, 214, 107, 0.35) !important;
}

.gradio-checkbox label,
.gradio-dropdown label,
.gradio-radio label,
.gradio-label,
.gradio-text {
    color: #e9d9a8 !important;
}

.gr-button {
    background: linear-gradient(135deg, #d4af37 0%, #b8860b 100%) !important;
    color: #1a1205 !important;
    border: 1px solid #f0d57a !important;
    font-weight: 700 !important;
    box-shadow: 0 0 10px rgba(212, 175, 55, 0.28) !important;
}

.gr-button:hover {
    background: linear-gradient(135deg, #f0c34a 0%, #c99713 100%) !important;
    color: #140e04 !important;
}

.tab-nav button,
.gradio-tabs button {
    background: #1c160a !important;
    color: #e8d7a3 !important;
    border: 1px solid #8d6f24 !important;
}

.tab-nav button.selected,
.gradio-tabs button.selected {
    background: linear-gradient(135deg, #b8860b 0%, #d4af37 100%) !important;
    color: #1a1205 !important;
}
"""

with gr.Blocks(title="JSON Refiner Advanced Edition", css=custom_css, theme=gr.themes.Soft()) as demo:

    gr.Markdown("# JSON Refiner Advanced Edition")

    state = gr.State({"items": [deepcopy(sample_json)], "idx": 0, "events": []})

    # Moved model loading to the top of the UI definition
    with gr.Row():
        btn_load_model = gr.Button("Load Granite Model")
        model_status = gr.Textbox(label="Model Status")

    with gr.Row():
        input_mode = gr.Radio(["JSON", "Key:Value"], value="JSON", label="Input Type")
        case_style = gr.Dropdown(
            ["keep", "snake_case", "camelCase", "PascalCase", "kebab-case"],
            value="camelCase",
            label="Case Style"
        )
        use_merge = gr.Checkbox(True, label="Merge")
        do_flatten = gr.Checkbox(False, label="Flatten")
        do_unflatten = gr.Checkbox(False, label="Unflatten")
        drop_nulls = gr.Checkbox(False, label="Remove Nulls")
        sort_keys = gr.Checkbox(False, label="Sort Keys")
        use_ai = gr.Checkbox(False, label="AI Summary")

    with gr.Row():
        input_text = gr.Textbox(
            lines=12,
            label="Input",
            value=json.dumps(sample_json, indent=2),
            elem_classes=["input_textbox"]
        )
        merge_text = gr.Textbox(lines=8, label="Merge JSON", elem_classes=["input_textbox"])

    schema_text = gr.Textbox(lines=6, label="Schema (optional)", value='{"type":"object"}', elem_classes=["input_textbox"])

    btn_process = gr.Button("Process", variant="primary")

    status = gr.Textbox(lines=6)

    with gr.Tabs():

        with gr.TabItem("JSON"):
            out_json = gr.Code(language="json", label="JSON", elem_classes=["output_codeblock"])

        with gr.TabItem("YAML"):
            out_yaml = gr.Code(language="yaml", label="YAML", elem_classes=["output_codeblock"])

        with gr.TabItem("CSV Flat"):
            out_csv = gr.Code(label="CSV", elem_classes=["output_codeblock"])

        with gr.TabItem("Report"):
            out_report = gr.Markdown()

        with gr.TabItem("Schema Hint"):
            out_schema = gr.Code(language="json", label="Schema", elem_classes=["output_codeblock"])

        with gr.TabItem("History"):
            out_history = gr.Code(language="json", label="History", elem_classes=["output_codeblock"])

    with gr.Row():
        file_output = gr.File(label="Download JSON")
        file_history = gr.File(label="Download History")
        file_report = gr.File(label="Download Report")

    # Moved upload options below output
    with gr.Row():
        input_file = gr.File(label="Upload Input JSON", file_types=[".json"])
        merge_file = gr.File(label="Upload Merge JSON", file_types=[".json"])
    with gr.Row():
        btn_load_input = gr.Button("Load Input File")
        btn_load_merge = gr.Button("Load Merge File")

    btn_load_input.click(read_json_file, [input_file], [input_text, status])
    btn_load_merge.click(read_json_file, [merge_file], [merge_text, status])

    btn_load_model.click(load_granite, outputs=model_status)

    btn_process.click(
        process_all,
        inputs=[
            input_mode,
            input_text,
            merge_text,
            schema_text,
            case_style,
            use_merge,
            do_flatten,
            do_unflatten,
            drop_nulls,
            sort_keys,
            use_ai,
            state,
        ],
        outputs=[
            out_json,
            out_yaml,
            out_csv,
            out_report,
            out_schema,
            status,
            state,
            file_output,
            file_history,
            file_report,
            out_history,
        ],
    )

/tmp/ipykernel_417/1371990221.py:83: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="JSON Refiner Advanced Edition", css=custom_css, theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_417/1371990221.py:83: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="JSON Refiner Advanced Edition", css=custom_css, theme=gr.themes.Soft()) as demo:


In [14]:
demo.launch(share=True, show_error=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3f3d3d044f4664e9f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
